# Analyse stratified split results

## Imports

In [1]:
import numpy as np
import pandas as pd
import json
from scipy import stats
from pathlib import Path

from deep_pianist_identification.utils import get_project_root

/home/huw-cheston/.cache/pypoetry/virtualenvs/deep-pianist-identification-2Pg5O36G-py3.12/lib/python3.12/site-packages/pretty_midi/instrument.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Logistic Regression

### Load in data

In [2]:
# Load up csv files
csvs = list((Path(get_project_root()) / "references/whitebox/stratified_split_experiments").rglob("**/*.csv"))

In [25]:
# Get accuracy values
lr_acc_base = [pd.read_csv(cs)["accuracy"].max() for cs in csvs if "_stratified_" not in str(cs)]
lr_acc_album = [pd.read_csv(cs)["accuracy"].max() for cs in csvs if "_stratified_album" in str(cs)][0]
lr_acc_composition = [pd.read_csv(cs)["accuracy"].max() for cs in csvs if "_stratified_composition" in str(cs)][0]

### Compute one-sample t-test

In [3]:
def t_test(test_sample: float, control_vals: list[float]) -> tuple[float, float]:
    control = np.array(control_vals)
    n = len(control)
    mean = np.mean(control)
    sd = np.std(control, ddof=1)  # sample std dev

    # T-statistic and one-tailed p-value (lower tail)
    t_stat = (test_sample - mean) / sd
    df = n - 1
    p_value = stats.t.cdf(t_stat, df=df)  # one-tailed, lower tail

    return t_stat, p_value


In [27]:
for name, val in zip(["Composition", "Album"], [lr_acc_composition, lr_acc_album]):
    t, p = t_test(val, lr_acc_base)
    print(f"LR --- {name}: accuracy = {val:.3f}, t = {t:.3f}, p = {p:.3f}")

LR --- Composition: accuracy = 0.798, t = 0.567, p = 0.711
LR --- Album: accuracy = 0.758, t = -1.475, p = 0.078


## ResNet results

In [27]:
js = (Path(get_project_root()) / "references/whitebox/stratified_split_experiments/resnet_stratified_results.json")
with open(js, "r") as f:
    resnet_data = json.load(f)


In [28]:
rn_acc_album = resnet_data["resnet50-jtd+pijama-augment-albumsplit"]
rn_acc_composition = resnet_data["resnet50-jtd+pijama-augment-compositionsplit"]
rn_acc_base = [v for k, v in resnet_data.items() if "-albumsplit" not in k and "-compositionsplit" not in k]
print(len(rn_acc_base))

20


In [30]:
for name, val in zip(["Composition", "Album"], [rn_acc_composition, rn_acc_album]):
    t, p = t_test(val, rn_acc_base)
    print(f"ResNet --- {name}: accuracy = {val:.3f}, t = {t:.3f}, p = {p:.3f}")

ResNet --- Composition: accuracy = 0.957, t = 0.764, p = 0.773
ResNet --- Album: accuracy = 0.920, t = -0.984, p = 0.169
